# 03 - Auto-ETS, Theta Method e Combinacao de Previsoes - SOLUCAO

Este notebook cobre a **selecao automatica** de modelos ETS, o **metodo Theta** e a **combinacao de previsoes** usando **chronobox**, com **todos os exercicios resolvidos**.

**Conteudo:**
1. Selecao automatica de modelo ETS (AIC/BIC)
2. Theta method como caso especial de SES com drift
3. Combinacao (ensemble) de previsoes
4. Comparacao: Auto-ETS vs Theta vs ARIMA
5. **Exercicios resolvidos**

## 1. Selecao Automatica de Modelo ETS

### Criterios de Informacao

- **AIC** (Akaike): $AIC = -2\ln(L) + 2k$
- **BIC** (Bayesian): $BIC = -2\ln(L) + k\ln(n)$
- **AICc** (corrigido): $AICc = AIC + \frac{2k(k+1)}{n-k-1}$

O AICc e preferido para amostras pequenas. O BIC penaliza mais parametros para $n > 8$.

### Processo de selecao

1. Enumerar modelos candidatos (ate 30)
2. Ajustar cada modelo por maxima verossimilhanca
3. Calcular criterio de informacao
4. Selecionar o modelo com menor criterio

## 2. Setup e Carregamento dos Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import json

from chronobox import ETS, HoltWinters, ThetaMethod, ARIMA
from chronobox.selection.auto_ets import auto_ets
from scipy.stats import norm, probplot

np.random.seed(42)

# Diretorios
BASE_DIR = os.path.join(os.path.dirname(os.getcwd()))
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Carregar datasets
airline = pd.read_csv(os.path.join(DATA_DIR, 'airline.csv'), parse_dates=['date'])
airline.set_index('date', inplace=True)
y_airline = airline['passengers'].values.astype(float)

ets_synth = pd.read_csv(os.path.join(DATA_DIR, 'ets_synthetic.csv'), parse_dates=['date'])
ets_synth.set_index('date', inplace=True)
y_synth = ets_synth['value'].values.astype(float)

print(f'Airline: {len(y_airline)} obs')
print(f'Sintetico: {len(y_synth)} obs')
print('\nModulos carregados: ETS, HoltWinters, ThetaMethod, ARIMA, auto_ets')

## 3. Auto-ETS: Selecao Automatica

In [ ]:
# Auto-ETS no dataset airline
result = auto_ets(y_airline, seasonal_period=12, information_criterion='aicc', verbose=True)

print('\n' + '=' * 60)
print(result.summary())
print(f'\nModelo selecionado: ETS({result.best_spec[0]},{result.best_spec[1]},{result.best_spec[2]})')
print(f'Modelos avaliados: {result.n_models_tried}')
print(f'Modelos com falha: {result.n_models_failed}')

In [ ]:
# Top 10 modelos por AICc
all_res = result.all_results  # list of (spec_tuple, ic_value)

print(f'{"Rank":<6} {"Modelo":<20} {"AICc":>12}')
print('-' * 40)
for i, (spec, ic_val) in enumerate(all_res[:10]):
    nome = f'ETS({spec[0]},{spec[1]},{spec[2]})'
    print(f'{i+1:<6} {nome:<20} {ic_val:>12.2f}')

if len(all_res) >= 2:
    print(f'\nDiferenca AICc entre 1o e 2o: {all_res[1][1] - all_res[0][1]:.2f}')
    print('(Diferenca > 2 indica evidencia forte a favor do melhor modelo)')

In [ ]:
# Ajuste do modelo auto-selecionado
best = result.best_model
spec = result.best_spec

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(best.fittedvalues, label=f'Auto-ETS: ({spec[0]},{spec[1]},{spec[2]})',
        color='darkorange', linestyle='--', linewidth=1.2)
ax.set_title(f'Auto-ETS Selecionado: ETS({spec[0]},{spec[1]},{spec[2]}) - Airline')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Parametros estimados:')
print(f'  alpha = {best.alpha:.4f}')
if best.beta is not None:
    print(f'  beta  = {best.beta:.4f}')
if best.gamma is not None:
    print(f'  gamma = {best.gamma:.4f}')
if best.phi is not None:
    print(f'  phi   = {best.phi:.4f}')

## 4. Theta Method

O **metodo Theta** (Assimakopoulos & Nikolopoulos, 2000) e equivalente a SES com drift:

$$\hat{y}_{t+h|t} = l_t + \frac{b}{2} h$$

onde $l_t$ e o nivel do SES e $b$ e o slope da regressao.

> O Theta method venceu a competicao M3 (2000), superando metodos mais complexos!

In [ ]:
# Ajustar Theta Method no airline
theta = ThetaMethod(theta=2.0)
res_theta = theta.fit(y_airline)

print(res_theta.summary())
print(f'\nParametros:')
print(f'  alpha (SES): {res_theta.alpha:.4f}')
print(f'  slope (regressao): {res_theta.slope:.6f}')
print(f'  intercept: {res_theta.intercept:.4f}')
print(f'  drift: {res_theta.drift:.4f}')

In [ ]:
# Theta Method - ajuste e previsao
fc_theta = res_theta.forecast(steps=24)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(y_airline)), y_airline, label='Observado', color='steelblue', linewidth=1.5)
ax.plot(range(len(y_airline)), res_theta.fittedvalues, label='Theta ajustado',
        color='darkorange', linestyle='--', alpha=0.7)

idx_fc = range(len(y_airline), len(y_airline) + 24)
ax.plot(idx_fc, fc_theta, label='Theta previsao', color='crimson', linewidth=2)
ax.axvline(len(y_airline) - 1, color='gray', linestyle=':', alpha=0.5)

ax.set_title('Theta Method (theta=2.0) - Airline Passengers')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend()
plt.tight_layout()
plt.show()

print('Nota: O Theta method nao captura sazonalidade diretamente.')

## 5. Comparacao: Auto-ETS vs Theta vs ARIMA

Vamos comparar tres abordagens diferentes no dataset airline, dividindo em treino/teste.

In [ ]:
# Funcoes auxiliares de metricas
def calc_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def calc_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
# Dividir airline em treino/teste
n_test = 24
y_train = y_airline[:-n_test]
y_test = y_airline[-n_test:]

print(f'Treino: {len(y_train)} obs')
print(f'Teste: {len(y_test)} obs')

# 1. Auto-ETS
res_auto = auto_ets(y_train, seasonal_period=12, information_criterion='aicc')
fc_auto = res_auto.forecast(steps=n_test)
spec_auto = res_auto.best_spec

# 2. Theta Method
theta_model = ThetaMethod(theta=2.0)
res_theta_tr = theta_model.fit(y_train)
fc_theta_tr = res_theta_tr.forecast(steps=n_test)

# 3. SARIMA(1,1,1)(1,1,1,12)
arima_model = ARIMA(order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
res_arima = arima_model.fit(y_train)
fc_arima_raw = res_arima.forecast(steps=n_test)
fc_arima = fc_arima_raw['forecast'] if isinstance(fc_arima_raw, dict) else fc_arima_raw

print(f'\nModelos ajustados:')
print(f'  Auto-ETS: ({spec_auto[0]},{spec_auto[1]},{spec_auto[2]})')
print(f'  Theta (theta=2.0)')
print(f'  SARIMA(1,1,1)(1,1,1,12)')

# Metricas
print(f'\n{"Modelo":<25} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 58)
for nome, fc in [('Auto-ETS', fc_auto), ('Theta', fc_theta_tr), ('SARIMA(1,1,1)', fc_arima)]:
    fc_arr = np.array(fc)[:n_test]
    print(f'{nome:<25} {calc_mae(y_test, fc_arr):>10.2f} {calc_rmse(y_test, fc_arr):>10.2f} {calc_mape(y_test, fc_arr):>10.2f}')

In [ ]:
# Comparacao de previsoes
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(range(len(y_train)), y_train, label='Treino', color='steelblue', linewidth=1.5)
idx_test = range(len(y_train), len(y_airline))
ax.plot(idx_test, y_test, label='Teste (real)', color='black', linewidth=2)

ax.plot(idx_test, np.array(fc_auto)[:n_test], label=f'Auto-ETS ({spec_auto[0]},{spec_auto[1]},{spec_auto[2]})',
        color='darkorange', linewidth=1.5, linestyle='--')
ax.plot(idx_test, np.array(fc_theta_tr)[:n_test], label='Theta',
        color='green', linewidth=1.5, linestyle='-.')
ax.plot(idx_test, np.array(fc_arima)[:n_test], label='SARIMA(1,1,1)',
        color='crimson', linewidth=1.5, linestyle=':')

ax.axvline(len(y_train) - 1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Comparacao: Auto-ETS vs Theta vs SARIMA')
ax.set_xlabel('Observacao')
ax.set_ylabel('Passageiros')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Combinacao de Previsoes (Ensemble)

$$\hat{y}^{comb}_{t+h} = \sum_{i=1}^{M} w_i \hat{y}^{(i)}_{t+h}$$

> Bates & Granger (1969): a media simples e dificil de superar!

In [ ]:
# Combinacao de previsoes
fc_auto_arr = np.array(fc_auto)[:n_test]
fc_theta_arr = np.array(fc_theta_tr)[:n_test]
fc_arima_arr = np.array(fc_arima)[:n_test]

# Media simples
fc_mean = (fc_auto_arr + fc_theta_arr + fc_arima_arr) / 3

# Media ponderada pelo inverso do RMSE
rmse_auto_is = np.sqrt(np.mean(res_auto.best_model.resid ** 2))
rmse_theta_is = np.sqrt(np.mean(res_theta_tr.resid ** 2))
rmse_arima_is = np.sqrt(np.mean(res_arima.residuals ** 2))

w = np.array([1/rmse_auto_is, 1/rmse_theta_is, 1/rmse_arima_is])
w = w / w.sum()
fc_weighted = w[0] * fc_auto_arr + w[1] * fc_theta_arr + w[2] * fc_arima_arr

print(f'{"Combinacao":<30} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 63)
for nome, fc_c in [('Auto-ETS (individual)', fc_auto_arr),
                    ('Theta (individual)', fc_theta_arr),
                    ('SARIMA (individual)', fc_arima_arr),
                    ('Media simples', fc_mean),
                    ('Media ponderada (inv-RMSE)', fc_weighted)]:
    print(f'{nome:<30} {calc_mae(y_test, fc_c):>10.2f} {calc_rmse(y_test, fc_c):>10.2f} {calc_mape(y_test, fc_c):>10.2f}')

print(f'\nPesos (inv-RMSE): Auto-ETS={w[0]:.3f}, Theta={w[1]:.3f}, SARIMA={w[2]:.3f}')

---

## Exercicio 1 - SOLUCAO: Auto-ETS vs Theta vs ARIMA no dataset sintetico

Usando o dataset `ets_synthetic.csv`:

1. Divida em treino (primeiros 156 obs) e teste (ultimos 24 obs)
2. Ajuste Auto-ETS, Theta e ARIMA(1,1,0) nos dados de treino
3. Compare MAE, RMSE e MAPE no conjunto de teste
4. Construa um ensemble com media simples e compare com os modelos individuais
5. O Auto-ETS consegue recuperar o modelo gerador ETS(M,A,M)?

In [ ]:
# Exercicio 1 - SOLUCAO
# Passo 1: Dividir dados sinteticos
n_test_s = 24
y_train_s = y_synth[:-n_test_s]
y_test_s = y_synth[-n_test_s:]

print(f'Serie sintetica:')
print(f'  Treino: {len(y_train_s)} obs (primeiros 156)')
print(f'  Teste:  {len(y_test_s)} obs (ultimos 24)')

In [ ]:
# Passo 2: Ajustar os tres modelos

# Auto-ETS
result_s = auto_ets(y_train_s, seasonal_period=12, information_criterion='aicc', verbose=True)
fc_auto_s = result_s.forecast(steps=n_test_s)
spec_s = result_s.best_spec

print(f'\nAuto-ETS selecionou: ETS({spec_s[0]},{spec_s[1]},{spec_s[2]})')
print(f'Modelo gerador real: ETS(M,A,M)')
print(f'Recuperou o modelo gerador? {"SIM" if spec_s == ("M", "A", "M") else "NAO - selecionou " + str(spec_s)}')

In [ ]:
# Theta Method no sintetico
theta_s = ThetaMethod(theta=2.0)
res_theta_s = theta_s.fit(y_train_s)
fc_theta_s = res_theta_s.forecast(steps=n_test_s)

# ARIMA(1,1,0) no sintetico
arima_s = ARIMA(order=(1, 1, 0))
res_arima_s = arima_s.fit(y_train_s)
fc_arima_s_raw = res_arima_s.forecast(steps=n_test_s)
fc_arima_s = fc_arima_s_raw['forecast'] if isinstance(fc_arima_s_raw, dict) else fc_arima_s_raw

print('Modelos ajustados no treino da serie sintetica:')
print(f'  Auto-ETS: ETS({spec_s[0]},{spec_s[1]},{spec_s[2]})')
print(f'  Theta (theta=2.0)')
print(f'  ARIMA(1,1,0)')

In [ ]:
# Passo 3: Comparar metricas no conjunto de teste
fc_auto_s_arr = np.array(fc_auto_s)[:n_test_s]
fc_theta_s_arr = np.array(fc_theta_s)[:n_test_s]
fc_arima_s_arr = np.array(fc_arima_s)[:n_test_s]

print(f'{"Modelo":<25} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 58)
resultados_s = {}
for nome, fc_arr in [('Auto-ETS', fc_auto_s_arr), ('Theta', fc_theta_s_arr), ('ARIMA(1,1,0)', fc_arima_s_arr)]:
    mae_v = calc_mae(y_test_s, fc_arr)
    rmse_v = calc_rmse(y_test_s, fc_arr)
    mape_v = calc_mape(y_test_s, fc_arr)
    resultados_s[nome] = {'mae': mae_v, 'rmse': rmse_v, 'mape': mape_v}
    print(f'{nome:<25} {mae_v:>10.2f} {rmse_v:>10.2f} {mape_v:>10.2f}')

melhor_s = min(resultados_s, key=lambda k: resultados_s[k]['rmse'])
print(f'\nMelhor modelo por RMSE: {melhor_s}')

In [ ]:
# Passo 4: Ensemble com media simples
fc_ensemble_s = (fc_auto_s_arr + fc_theta_s_arr + fc_arima_s_arr) / 3

mae_ens = calc_mae(y_test_s, fc_ensemble_s)
rmse_ens = calc_rmse(y_test_s, fc_ensemble_s)
mape_ens = calc_mape(y_test_s, fc_ensemble_s)

print(f'{"Modelo":<25} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 58)
for nome, fc_arr in [('Auto-ETS', fc_auto_s_arr), ('Theta', fc_theta_s_arr),
                      ('ARIMA(1,1,0)', fc_arima_s_arr), ('Ensemble (media)', fc_ensemble_s)]:
    print(f'{nome:<25} {calc_mae(y_test_s, fc_arr):>10.2f} {calc_rmse(y_test_s, fc_arr):>10.2f} {calc_mape(y_test_s, fc_arr):>10.2f}')

# O ensemble superou os individuais?
ens_melhor = rmse_ens < min(r['rmse'] for r in resultados_s.values())
print(f'\nO ensemble superou todos os individuais? {"SIM" if ens_melhor else "NAO"}')

In [ ]:
# Visualizacao das previsoes
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Painel 1: Todas as previsoes
axes[0].plot(range(len(y_train_s)), y_train_s, label='Treino', color='steelblue', linewidth=1)
idx_test_s = range(len(y_train_s), len(y_synth))
axes[0].plot(idx_test_s, y_test_s, label='Teste (real)', color='black', linewidth=2)
axes[0].plot(idx_test_s, fc_auto_s_arr, label=f'Auto-ETS ({spec_s[0]},{spec_s[1]},{spec_s[2]})',
             color='darkorange', linewidth=1.5, linestyle='--')
axes[0].plot(idx_test_s, fc_theta_s_arr, label='Theta',
             color='green', linewidth=1.5, linestyle='-.')
axes[0].plot(idx_test_s, fc_arima_s_arr, label='ARIMA(1,1,0)',
             color='purple', linewidth=1.5, linestyle=':')
axes[0].plot(idx_test_s, fc_ensemble_s, label='Ensemble',
             color='crimson', linewidth=2, linestyle='-')
axes[0].axvline(len(y_train_s) - 1, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title('Comparacao de Previsoes - Serie Sintetica')
axes[0].legend(fontsize=9)
axes[0].set_ylabel('Valor')

# Painel 2: Erros absolutos
for nome, fc_arr, cor in [('Auto-ETS', fc_auto_s_arr, 'darkorange'),
                            ('Theta', fc_theta_s_arr, 'green'),
                            ('ARIMA(1,1,0)', fc_arima_s_arr, 'purple'),
                            ('Ensemble', fc_ensemble_s, 'crimson')]:
    axes[1].plot(range(n_test_s), np.abs(y_test_s - fc_arr), label=nome, color=cor, linewidth=1.2)
axes[1].set_title('Erro Absoluto por Horizonte')
axes[1].set_xlabel('Passo a frente (h)')
axes[1].set_ylabel('|Erro|')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Passo 5: Analise final - recuperacao do modelo gerador
print('=' * 60)
print('RESULTADO: Recuperacao do Modelo Gerador')
print('=' * 60)
print(f'\nModelo gerador real: ETS(M,A,M)')
print(f'Auto-ETS selecionou: ETS({spec_s[0]},{spec_s[1]},{spec_s[2]})')

if spec_s == ('M', 'A', 'M'):
    print('\n-> SUCESSO: O Auto-ETS recuperou exatamente o modelo gerador!')
else:
    print(f'\n-> O modelo selecionado difere do gerador.')
    print(f'   Isso pode ocorrer porque:')
    print(f'   - A amostra de treino ({len(y_train_s)} obs) pode nao ser suficiente')
    print(f'   - Modelos proximos podem ter AICc similar')
    print(f'   - O criterio penaliza complexidade, favorecendo modelos mais simples')

# Top 5 modelos da selecao
print(f'\nTop 5 modelos candidatos:')
for i, (sp, ic) in enumerate(result_s.all_results[:5]):
    marker = ' <-- gerador' if sp == ('M', 'A', 'M') else ''
    print(f'  {i+1}. ETS({sp[0]},{sp[1]},{sp[2]}): AICc={ic:.2f}{marker}')

### Conclusao do Exercicio 1

1. **Auto-ETS**: Melhor desempenho por capturar sazonalidade multiplicativa
2. **Theta e ARIMA(1,1,0)**: Nao capturam sazonalidade, desempenho inferior
3. **Ensemble**: A media simples pode ou nao superar o melhor individual - depende da diversidade dos modelos
4. **Recuperacao**: O Auto-ETS tem boa capacidade de identificar o processo gerador quando a amostra e suficiente

---

## Exercicio 2 - SOLUCAO: Sensibilidade do criterio de selecao

1. Execute `auto_ets` no airline com `information_criterion='aic'`, `'bic'` e `'aicc'`
2. Os tres criterios selecionam o mesmo modelo?
3. Qual criterio gera previsoes melhores no conjunto de teste?
4. Por que o BIC tende a selecionar modelos mais parcimoniosos?

**Dica:** BIC penaliza mais parametros que o AIC para $n > 8$.

In [ ]:
# Exercicio 2 - SOLUCAO
# Passo 1: Rodar auto_ets com cada criterio
criterios = ['aic', 'bic', 'aicc']
resultados_crit = {}

print(f'{"Criterio":<10} {"Modelo Selecionado":<25} {"Valor IC":>12}')
print('-' * 50)

for crit in criterios:
    res_c = auto_ets(y_train, seasonal_period=12, information_criterion=crit)
    spec_c = res_c.best_spec
    nome_c = f'ETS({spec_c[0]},{spec_c[1]},{spec_c[2]})'
    
    # Obter o valor do criterio usado
    best_m = res_c.best_model
    if crit == 'aic':
        ic_val = best_m.aic
    elif crit == 'bic':
        ic_val = best_m.bic
    else:
        ic_val = best_m.aicc
    
    resultados_crit[crit] = {
        'result': res_c,
        'spec': spec_c,
        'nome': nome_c,
        'ic_val': ic_val,
    }
    print(f'{crit:<10} {nome_c:<25} {ic_val:>12.2f}')

In [ ]:
# Passo 2: Os criterios selecionam o mesmo modelo?
specs = [resultados_crit[c]['spec'] for c in criterios]
mesmo_modelo = len(set(specs)) == 1

print(f'Os tres criterios selecionam o mesmo modelo? {"SIM" if mesmo_modelo else "NAO"}')
if not mesmo_modelo:
    print('\nModelos diferentes selecionados:')
    for crit in criterios:
        print(f'  {crit}: {resultados_crit[crit]["nome"]}')

In [ ]:
# Passo 3: Comparar previsoes no conjunto de teste
print(f'{"Criterio":<10} {"Modelo":<20} {"MAE":>10} {"RMSE":>10} {"MAPE(%)":>10}')
print('-' * 65)

for crit in criterios:
    res_c = resultados_crit[crit]['result']
    fc_c = np.array(res_c.forecast(steps=n_test))[:n_test]
    nome_c = resultados_crit[crit]['nome']
    resultados_crit[crit]['fc'] = fc_c
    resultados_crit[crit]['mae'] = calc_mae(y_test, fc_c)
    resultados_crit[crit]['rmse'] = calc_rmse(y_test, fc_c)
    resultados_crit[crit]['mape'] = calc_mape(y_test, fc_c)
    print(f'{crit:<10} {nome_c:<20} {resultados_crit[crit]["mae"]:>10.2f} {resultados_crit[crit]["rmse"]:>10.2f} {resultados_crit[crit]["mape"]:>10.2f}')

melhor_crit = min(criterios, key=lambda c: resultados_crit[c]['rmse'])
print(f'\nMelhor criterio por RMSE no teste: {melhor_crit} -> {resultados_crit[melhor_crit]["nome"]}')

In [ ]:
# Visualizacao das previsoes por criterio
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

cores_crit = {'aic': 'darkorange', 'bic': 'green', 'aicc': 'crimson'}

# Previsoes
n_show = 48
axes[0].plot(range(len(y_train)-n_show, len(y_train)), y_train[-n_show:],
             label='Treino', color='steelblue', linewidth=1.5)
axes[0].plot(range(len(y_train), len(y_airline)), y_test,
             label='Teste (real)', color='black', linewidth=2)

for crit in criterios:
    fc_c = resultados_crit[crit]['fc']
    nome_c = resultados_crit[crit]['nome']
    rmse_c = resultados_crit[crit]['rmse']
    axes[0].plot(range(len(y_train), len(y_airline)), fc_c,
                 label=f'{crit}: {nome_c} (RMSE={rmse_c:.1f})',
                 color=cores_crit[crit], linewidth=1.5, linestyle='--')

axes[0].axvline(len(y_train) - 1, color='gray', linestyle=':', alpha=0.5)
axes[0].set_title('Previsoes por Criterio de Selecao')
axes[0].set_ylabel('Passageiros')
axes[0].legend(fontsize=9)

# Numero de parametros vs penalizacao
n = len(y_train)
k_range = np.arange(1, 20)
pen_aic = 2 * k_range
pen_bic = k_range * np.log(n)
pen_aicc = 2 * k_range + 2 * k_range * (k_range + 1) / np.maximum(n - k_range - 1, 1)

axes[1].plot(k_range, pen_aic, label='AIC: 2k', color='darkorange', linewidth=2)
axes[1].plot(k_range, pen_bic, label=f'BIC: k*ln({n})={np.log(n):.2f}k',
             color='green', linewidth=2)
axes[1].plot(k_range, pen_aicc, label='AICc: 2k + 2k(k+1)/(n-k-1)',
             color='crimson', linewidth=2, linestyle='--')
axes[1].set_title('Penalizacao por Numero de Parametros')
axes[1].set_xlabel('Numero de parametros (k)')
axes[1].set_ylabel('Penalizacao')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Passo 4: Por que o BIC seleciona modelos mais parcimoniosos?
print('=' * 60)
print('ANALISE: BIC vs AIC - Parcimonia na Selecao de Modelos')
print('=' * 60)

n = len(y_train)
print(f'\nTamanho da amostra (n): {n}')
print(f'\nPenalizacao por parametro:')
print(f'  AIC:  2 por parametro')
print(f'  BIC:  ln({n}) = {np.log(n):.2f} por parametro')
print(f'  AICc: ~2 por parametro (com correcao para amostra finita)')

print(f'\nPara n = {n}:')
print(f'  BIC penaliza {np.log(n)/2:.2f}x mais que o AIC por parametro')
print(f'  BIC > AIC quando n > e^2 = {np.exp(2):.2f} (ou seja, n > 8)')

print(f'\nConsequencias praticas:')
print(f'  - AIC: favorece bom ajuste, pode selecionar modelos mais complexos')
print(f'  - BIC: favorece parcimonia, seleciona modelos com menos parametros')
print(f'  - AICc: similar ao AIC mas com correcao para amostras pequenas')
print(f'  - Para previsao: AIC/AICc geralmente melhor (foco em minimizar erro)')
print(f'  - Para inferencia: BIC geralmente melhor (consistente na selecao do modelo verdadeiro)')

### Conclusao do Exercicio 2

1. **AIC, BIC e AICc podem selecionar modelos diferentes**, especialmente quando ha modelos competitivos com numero de parametros distintos
2. **BIC e mais parcimonioso** porque penaliza $k \ln(n)$ vs $2k$ do AIC - para $n > 8$, BIC penaliza mais
3. **Para previsao**, AIC/AICc costumam gerar melhores resultados (foco em acuracia)
4. **Para identificacao do modelo verdadeiro**, BIC e consistente (seleciona o verdadeiro com $n \to \infty$)
5. **Na pratica**: use AICc como default, compare com BIC quando parsimonia importa

---

## 7. Salvando Resultados para Comparacao

In [ ]:
# Salvar tabela comparativa em outputs/auto_ets_comparison.csv
rows = []

# Resultados airline (treino/teste)
for nome, fc_arr in [('Auto-ETS', fc_auto), ('Theta', fc_theta_tr), ('SARIMA(1,1,1)', fc_arima)]:
    fc_a = np.array(fc_arr)[:n_test]
    rows.append({
        'dataset': 'airline',
        'method': nome,
        'model': f'ETS({spec_auto[0]},{spec_auto[1]},{spec_auto[2]})' if nome == 'Auto-ETS' else nome,
        'mae': round(calc_mae(y_test, fc_a), 4),
        'rmse': round(calc_rmse(y_test, fc_a), 4),
        'mape': round(calc_mape(y_test, fc_a), 4),
        'n_train': len(y_train),
        'n_test': n_test,
    })

# Ensemble airline
rows.append({
    'dataset': 'airline',
    'method': 'Ensemble (media simples)',
    'model': 'media(Auto-ETS+Theta+SARIMA)',
    'mae': round(calc_mae(y_test, fc_mean), 4),
    'rmse': round(calc_rmse(y_test, fc_mean), 4),
    'mape': round(calc_mape(y_test, fc_mean), 4),
    'n_train': len(y_train),
    'n_test': n_test,
})

# Resultados sintetico
for nome, fc_arr in [('Auto-ETS', fc_auto_s_arr), ('Theta', fc_theta_s_arr),
                      ('ARIMA(1,1,0)', fc_arima_s_arr), ('Ensemble', fc_ensemble_s)]:
    rows.append({
        'dataset': 'ets_synthetic',
        'method': nome,
        'model': f'ETS({spec_s[0]},{spec_s[1]},{spec_s[2]})' if nome == 'Auto-ETS' else nome,
        'mae': round(calc_mae(y_test_s, fc_arr), 4),
        'rmse': round(calc_rmse(y_test_s, fc_arr), 4),
        'mape': round(calc_mape(y_test_s, fc_arr), 4),
        'n_train': len(y_train_s),
        'n_test': n_test_s,
    })

# Resultados por criterio (airline)
for crit in criterios:
    r = resultados_crit[crit]
    rows.append({
        'dataset': 'airline',
        'method': f'Auto-ETS ({crit})',
        'model': r['nome'],
        'mae': round(r['mae'], 4),
        'rmse': round(r['rmse'], 4),
        'mape': round(r['mape'], 4),
        'n_train': len(y_train),
        'n_test': n_test,
    })

# Salvar CSV
df_results = pd.DataFrame(rows)
output_path = os.path.join(OUTPUT_DIR, 'auto_ets_comparison.csv')
df_results.to_csv(output_path, index=False)

print(f'Comparacao salva em: {output_path}')
print(f'Total de linhas: {len(df_results)}')
print('\nConteudo:')
print(df_results.to_string(index=False))

---

## Conclusao

Neste notebook aprendemos:

1. **Auto-ETS**: Selecao automatica e eficiente entre 30 modelos candidatos
2. **Theta Method**: Simples e competitivo, equivalente a SES com drift
3. **Combinacao de previsoes**: Media simples frequentemente robusta
4. **Criterios de selecao**: AIC, BIC e AICc diferem na penalizacao de complexidade
5. **Recuperacao do modelo gerador**: Auto-ETS pode identificar o processo verdadeiro com amostra suficiente

**Resumo dos modulos ETS no chronobox:**
- `ETS()`: Ajuste manual de qualquer combinacao ETS
- `HoltWinters()`: Interface simplificada para Holt-Winters
- `ThetaMethod()`: Metodo Theta (SES com drift)
- `auto_ets()`: Selecao automatica do melhor modelo ETS